In [0]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
spark = SparkSession.builder.appName("Banking Data Analysis Gold Layer").getOrCreate()

In [0]:
base_path = "/Volumes/workspace/default/banking_data_analysis/"

customers = spark.read.format("delta").load(base_path + "delta_customers").alias("c")
accounts = spark.read.format("delta").load(base_path + "delta_accounts").alias("a")
transactions = spark.read.format("delta").load(base_path + "delta_transactions").alias("t")
loans = spark.read.format("delta").load(base_path + "delta_loans").alias("l")
branches = spark.read.format("delta").load(base_path + "delta_branches").alias("b")

In [0]:
customers.printSchema()
accounts.printSchema()
transactions.printSchema()
loans.printSchema()
branches.printSchema()

#### Create Master Table

In [0]:
gold_df = customers \
    .join(accounts, customers["customer_id"] == accounts["customer_id"], "left") \
    .join(transactions, accounts["account_id"] == transactions["account_id"], "left") \
    .join(loans, customers["customer_id"] == loans["customer_id"], "left") \
    .join(branches, accounts["branch_id"] == branches["branch_id"], "left")

display(gold_df)

#### Fact Transactions

In [0]:
fact_transactions = gold_df.select(
    "transaction_id",
    "account_id",
    "customer_id",
    "branch_id",
    "amount",
    "transaction_type",
    "transaction_date"
)

#### Customer KPI

In [0]:
from pyspark.sql.functions import sum, avg, count

kpi_customer_summary = gold_df.groupBy(col("c.customer_id")).agg(
    sum(col("a.balance")).alias("total_balance"),
    sum(col("t.amount")).alias("total_transaction_amount"),
    avg(col("a.balance")).alias("avg_balance"),
    count(col("t.transaction_id")).alias("transaction_count")
)

#### Branch Performance

In [0]:
kpi_branch_performance = gold_df.groupBy(col("a.branch_id")).agg(
    sum(col("t.amount")).alias("total_transactions"),
    count(col("t.transaction_id")).alias("transaction_count"),
    avg(col("a.balance")).alias("avg_balance")
)

#### Loan Analysis

In [0]:
kpi_loan_analysis = gold_df.groupBy(col("l.loan_type")).agg(
    count("*").alias("total_loans"),
    avg(col("l.loan_amount")).alias("avg_loan_amount")
)

#### Transaction Summary

In [0]:
kpi_transaction_summary = gold_df.groupBy(col("t.transaction_type")).agg(
    sum(col("t.amount")).alias("total_amount"),
    count("*").alias("transaction_count")
)

#### Customer Segmentation

In [0]:
from pyspark.sql.functions import when

customer_segmentation = gold_df.withColumn(
    "segment",
    when(col("a.balance") > 50000, "High Value")
    .when(col("a.balance") > 20000, "Medium Value")
    .otherwise("Low Value")
)

#### Top Customers

In [0]:
top_customers = gold_df.orderBy(col("a.balance").desc()).limit(10)

#### Monthly Trends

In [0]:
from pyspark.sql.functions import month, year

monthly_trends = gold_df.groupBy(
    year(col("t.transaction_date")).alias("year"),
    month(col("t.transaction_date")).alias("month")
).agg(
    sum(col("t.amount")).alias("monthly_revenue"),
    count(col("t.transaction_id")).alias("transactions")
)

#### High Risk Customers

In [0]:
high_risk_customers = gold_df.filter(
    (col("l.loan") == "yes") & (col("a.balance") < 1000)
)